# Isoglot 06: Repair B, safety-preserving fine-tuning

Fine-tunes Tiny Aya on notebook 05's CuratorKIT exports with SafeTune's **LISA** trainer, which
alternates safety batches (refusals to harmful requests in Hausa, Yoruba, Igbo and English) with
task batches (helpful answers to benign requests, many of which look risky), under a LoRA adapter.
The merged model then answers the 1,200 TukaBench prompts with the same resumable pipeline as
notebook 01. Judge with notebook 02, `VARIANT = "harden"`.

**Before running:** Accelerator **GPU T4 x2** (the base model is split across both cards),
Internet **on**, Secret **`HF_TOKEN`** ticked. **Save Version -> Save & Run All (Commit)**.
About 2.5 hours.


In [1]:
%%capture
!pip install -q "isoglot[gpu,mechanism] @ git+https://github.com/nftkingiii/isoglot"
# Kaggle ships torchao 0.10, which peft rejects when it wraps the model in LoRA
# (SafeTune's notes: uninstall torchao before importing SafeTune)
!pip uninstall -y -q torchao


In [2]:
import os, json, time, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # before torch touches CUDA
import torch
from pathlib import Path
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import HfApi, snapshot_download, whoami
api = HfApi()
USER = whoami()["name"]
REPO = f"{USER}/isoglot-runs"
WORK = Path("/kaggle/working")
snapshot_download(REPO, repo_type="dataset", local_dir=WORK, allow_patterns=["traindata/curated/**"])
CUR = WORK / "traindata" / "curated"
print(sorted(p.relative_to(CUR).as_posix() for p in CUR.rglob("*") if p.is_file())[:20])
print(torch.cuda.device_count(), "GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

['benign-input.jsonl', 'benign/README.md', 'benign/checksums.txt', 'benign/dataset_card.md', 'benign/lexsi_provenance.json', 'benign/manifest.json', 'benign/rejected.jsonl', 'benign/train/sft_messages.jsonl', 'benign/train/sft_sharegpt.jsonl', 'benign/validation/sft_messages.jsonl', 'benign/validation/sft_sharegpt.jsonl', 'harmful-input.jsonl', 'harmful/README.md', 'harmful/checksums.txt', 'harmful/dataset_card.md', 'harmful/lexsi_provenance.json', 'harmful/manifest.json', 'harmful/rejected.jsonl', 'harmful/train/sft_messages.jsonl', 'harmful/train/sft_sharegpt.jsonl']
2 GPUs: ['Tesla T4', 'Tesla T4']


## Train (SafeTune LISA on the CuratorKIT exports)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from safetune.runner import harden
from isoglot.patches import patch_lisa_multi_device

patch_lisa_multi_device()  # SafeTune 0.1.12's LISA sums terms across GPUs without moving them

BASE = "CohereLabs/tiny-aya-global"
tok = AutoTokenizer.from_pretrained(BASE)
# Frozen base in fp16 (6.7 GB) split evenly over both T4s, so each keeps ~10 GB for
# activations. peft keeps the trainable LoRA weights in fp32 (autocast_adapter_dtype),
# so fp16 mixed-precision training works. An fp32 base filled GPU 0 and ran out of memory.
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto",
                                             max_memory={0: "4GiB", 1: "4GiB"})
print({str(d) for d in model.hf_device_map.values()}, "| GPU0", round(torch.cuda.memory_allocated(0)/2**30, 1), "GB,",
      "GPU1", round(torch.cuda.memory_allocated(1)/2**30, 1), "GB")

trainer = harden.LisaTrainer(model, tok, epochs=2, batch_size=2, lr=1e-4, fp16=True,
                             lisa_rho=0.1, lisa_warmup_steps=10,
                             lisa_alignment_step=20, lisa_finetune_step=20)
t0 = time.time()
merged = trainer.train(str(CUR / "benign"), dataset_config="sft_messages",
                       safety_dataset=str(CUR / "harmful"), safety_dataset_config="sft_messages")
print(f"trained in {(time.time()-t0)/60:.1f} min -> {merged}")


I0000 00:00:1794418649.003892      22 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 21.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

... (9 outputs left out)

Generating validation split: 0 examples [00:00, ? examples/s]

Step,Training Loss
10,1.045715
20,1.199831
30,1.934436
40,2.086240
50,0.852817
60,0.725064
70,1.834561
80,2.069564
90,1.069102
100,0.920153


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

trained in 13.4 min -> /kaggle/working/results/checkpoints/lisa


## Save a half-precision copy and push it (private)

In [4]:
del trainer, model; gc.collect(); torch.cuda.empty_cache()
OUT_MODEL = WORK / "isoglot-tiny-aya-lisa"
m = AutoModelForCausalLM.from_pretrained(merged, dtype=torch.float16)
m.save_pretrained(OUT_MODEL); tok.save_pretrained(OUT_MODEL)
del m; gc.collect()
for f in Path(merged).glob("*provenance*.json"):
    (OUT_MODEL / f.name).write_bytes(f.read_bytes())
MODEL_REPO = f"{USER}/isoglot-tiny-aya-lisa"
api.create_repo(MODEL_REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=str(OUT_MODEL), repo_id=MODEL_REPO)
print("pushed", MODEL_REPO)


[transformers] The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

pushed NFTKING/isoglot-tiny-aya-lisa


## Answer all 1,200 TukaBench prompts with the repaired model (resumable)

In [5]:
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import EntryNotFoundError
from isoglot import LANGS, SETS
from isoglot.generate import hf_model, run_chunks

VARIANT = "harden"
RUNS = WORK / "runs"
def remote(path): return path.relative_to(RUNS).as_posix()
def fetch(path):
    try:
        hf_hub_download(REPO, remote(path), repo_type="dataset", local_dir=RUNS); print("restored", remote(path))
    except EntryNotFoundError:
        pass
def publish(path):
    api.upload_file(path_or_fileobj=str(path), path_in_repo=remote(path), repo_id=REPO, repo_type="dataset")

gen = hf_model(str(OUT_MODEL), batch_size=8)
t0 = time.time()
answers = run_chunks(gen, MODEL_REPO, RUNS, sets=SETS, langs=LANGS, variant=VARIANT,
                     max_tokens=512, fetch=fetch, publish=publish)
publish(answers)
print(f"done in {(time.time()-t0)/60:.1f} min ->", REPO, remote(answers))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

eng.csv:   0%|          | 0.00/17.8k [00:00<?, ?B/s]

done afri-jbb-harm/eng: 100 answers, 0 empty, 1.5 min


hau.csv:   0%|          | 0.00/19.5k [00:00<?, ?B/s]

done afri-jbb-harm/hau: 100 answers, 0 empty, 3.1 min


yor.csv:   0%|          | 0.00/23.2k [00:00<?, ?B/s]

... (16 outputs left out)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


done afri-jbb-benign/yor: 100 answers, 0 empty, 7.2 min


ibo.csv:   0%|          | 0.00/17.8k [00:00<?, ?B/s]

done afri-jbb-benign/ibo: 100 answers, 0 empty, 5.9 min
done in 53.9 min -> NFTKING/isoglot-runs harden/answers.jsonl
